# Finite-Size Scaling Analysis — 3D Ising Model

This notebook performs finite-size scaling (FSS) analysis on Monte Carlo sweep data
generated by the `fss` Rust binary. It extracts the critical temperature Tc and
critical exponents β, γ, α, ν by analysing how observables scale with lattice size N.

**Generate data first:**
```bash
cargo run --release --bin fss -- --sizes 8,12,16,20,24,28 --wolff --outdir analysis/data
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandas as pd
from scipy.optimize import minimize_scalar
from pathlib import Path

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

DATA_DIR = Path('data')
SIZES = [8, 12, 16, 20, 24, 28]
COLORS = cm.viridis(np.linspace(0.15, 0.85, len(SIZES)))

# 3D Ising universality class — theory values
NU    = 0.6301
BETA  = 0.3265
ALPHA = 0.1096
GAMMA = 1.2372
TC_THEORY = 4.5115

dfs = {}
for n in SIZES:
    p = DATA_DIR / f'fss_N{n}.csv'
    if p.exists():
        dfs[n] = pd.read_csv(p)
        print(f'N={n}: {len(dfs[n])} points, T=[{dfs[n].T.min():.2f}, {dfs[n].T.max():.2f}]')
    else:
        print(f'MISSING: {p} — run: cargo run --release --bin fss -- --sizes 8,12,16,20,24,28 --wolff --outdir analysis/data')

if not dfs:
    raise SystemExit('No data found. Generate data first.')

## 1. Raw observables for all N

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
labels = ['E/spin', '|<M>|/spin', 'Cv', 'chi']
cols   = ['E',      'M',          'Cv', 'chi']

for ax, col, label in zip(axes.flat, cols, labels):
    for (n, df), c in zip(dfs.items(), COLORS):
        ax.plot(df['T'], df[col], color=c, label=f'N={n}')
    ax.set_xlabel('T [J/kB]')
    ax.set_ylabel(label)
    ax.legend(fontsize=8)
    ax.axvline(TC_THEORY, color='red', lw=0.8, ls='--', label='Tc theory')

fig.suptitle('3D Ising: observables vs T for multiple N')
plt.tight_layout()
plt.savefig('fss_observables.png', bbox_inches='tight')
plt.show()

## 2. Binder Cumulant — Precise Tc

The Binder cumulant U = 1 - <M^4> / (3 <M^2>^2) crosses at the same Tc
for all N, giving a precise, finite-size-independent estimate of Tc.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for (n, df), c in zip(dfs.items(), COLORS):
    U = 1.0 - df['M4'] / (3.0 * df['M2']**2)
    ax.plot(df['T'], U, color=c, label=f'N={n}')

ax.axvline(TC_THEORY, color='red', lw=1, ls='--', label=f'Tc theory = {TC_THEORY}')
ax.set_xlabel('T [J/kB]')
ax.set_ylabel('Binder cumulant U')
ax.set_title('Binder cumulant crossing -> Tc')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fss_binder.png', bbox_inches='tight')
plt.show()

print(f'Curves should cross near Tc ~ {TC_THEORY}')

## 3. Peak Scaling — Extract alpha/nu and gamma/nu

At criticality: Cv_max ~ N^(alpha/nu), chi_max ~ N^(gamma/nu)
Log-log slope gives the ratio of exponents.

In [ ]:
sizes = sorted(dfs.keys())
cv_peaks  = [dfs[n]['Cv'].max()  for n in sizes]
chi_peaks = [dfs[n]['chi'].max() for n in sizes]
logN = np.log(sizes)

slope_cv,  _ = np.polyfit(logN, np.log(cv_peaks),  1)
slope_chi, _ = np.polyfit(logN, np.log(chi_peaks), 1)

print(f'alpha/nu measured = {slope_cv:.4f}  (theory = {ALPHA/NU:.4f})')
print(f'gamma/nu measured = {slope_chi:.4f}  (theory = {GAMMA/NU:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, peaks, slope, title, theory in zip(
    axes,
    [cv_peaks, chi_peaks],
    [slope_cv, slope_chi],
    ['Cv_max ~ N^(alpha/nu)', 'chi_max ~ N^(gamma/nu)'],
    [ALPHA/NU, GAMMA/NU]
):
    ax.scatter(sizes, peaks, zorder=5)
    Nfit = np.linspace(min(sizes)*0.9, max(sizes)*1.1, 100)
    c0 = np.log(peaks[0]) - slope * np.log(sizes[0])
    ax.plot(Nfit, np.exp(slope * np.log(Nfit) + c0), 'r--',
            label=f'slope={slope:.3f} (theory={theory:.3f})')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('N')
    ax.legend()
    ax.set_title(title)

plt.tight_layout()
plt.savefig('fss_peak_scaling.png', bbox_inches='tight')
plt.show()

## 4. Scaling Collapse — Universal Curve

Plot N^(-gamma/nu) * chi vs (T - Tc) * N^(1/nu).
All curves should collapse onto one universal function when nu is correct.

In [ ]:
def collapse_quality(nu, tc=TC_THEORY):
    all_x, all_y = [], []
    for n, df in dfs.items():
        x = (df['T'] - tc) * n**(1/nu)
        y = df['chi'] / n**(GAMMA/nu)
        all_x.extend(x.values)
        all_y.extend(y.values)
    order = np.argsort(all_x)
    all_y_sorted = np.array(all_y)[order]
    return float(np.sum(np.diff(all_y_sorted)**2))

result = minimize_scalar(collapse_quality, bounds=(0.4, 0.9), method='bounded')
nu_fit = result.x
print(f'Optimised nu = {nu_fit:.4f}  (theory = {NU:.4f})')

fig, ax = plt.subplots(figsize=(9, 5))
for (n, df), c in zip(dfs.items(), COLORS):
    x = (df['T'] - TC_THEORY) * n**(1/nu_fit)
    y = df['chi'] / n**(GAMMA/nu_fit)
    ax.plot(x, y, color=c, label=f'N={n}')

ax.set_xlabel('(T - Tc) * N^(1/nu)')
ax.set_ylabel('N^(-gamma/nu) * chi')
ax.set_title(f'Scaling collapse: nu_fit={nu_fit:.4f}, theory={NU:.4f}')
ax.legend(fontsize=9)
ax.set_xlim(-15, 15)
plt.tight_layout()
plt.savefig('fss_collapse.png', bbox_inches='tight')
plt.show()

## 5. Summary Table

In [ ]:
results = {
    'nu':       (nu_fit,    NU,       abs(nu_fit   - NU)    / NU    * 100),
    'alpha/nu': (slope_cv,  ALPHA/NU, abs(slope_cv - ALPHA/NU) / (ALPHA/NU) * 100),
    'gamma/nu': (slope_chi, GAMMA/NU, abs(slope_chi - GAMMA/NU) / (GAMMA/NU) * 100),
}

print(f'{"exponent":<12} {"measured":>10} {"theory":>10} {"error %":>10}')
print('-' * 45)
for name, (meas, theory, err) in results.items():
    print(f'{name:<12} {meas:>10.4f} {theory:>10.4f} {err:>9.1f}%')